### Installation of DeepEval

In [ ]:
# !pip install -U deepeval

In [9]:
!pip show deepeval

Name: deepeval
Version: 3.6.9
Summary: The LLM Evaluation Framework
Home-page: https://github.com/confident-ai/deepeval
Author: Jeffrey Ip
Author-email: jeffreyip@confident-ai.com
License: Apache-2.0
Location: C:\Users\Admin\Documents\GEN_AI\LLM_Evaluation\Test_AI\Dev\.venv\Lib\site-packages
Requires: aiohttp, anthropic, click, google-genai, grpcio, jinja2, nest_asyncio, ollama, openai, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, portalocker, posthog, pydantic, pydantic-settings, pyfiglet, pytest, pytest-asyncio, pytest-repeat, pytest-rerunfailures, pytest-xdist, python-dotenv, requests, rich, sentry-sdk, setuptools, tabulate, tenacity, tqdm, typer, wheel
Required-by: 


### Creating Confident AI login

In [ ]:
### Using Python

### Note - The API Key here should be Project API Key

import deepeval

deepeval.login("")


🎉🥳 Congratulations! You've successfully logged in! 🙌

In [ ]:
### Using CLI

#!deepeval login --confident-api-key YOUR_API_KEY


In [ ]:
# from dotenv import load_dotenv

# load = load_dotenv('./../.env')

In [ ]:
### TO be executed if you are using Google Colab

# from google.colab import userdata
# userdata.get('OPENAI_API_KEY')

In [ ]:
### Set the environment variable - To be used in Google Colab

# import os
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
# os.environ['OPENAI_API_KEY']

In [3]:
### If you are using .env file
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

In [ ]:
import os
os.environ['OPENAI_API_KEY']

In [ ]:
### Cli Help Command for Deepeval

!deepeval --help

                                                                               
 Usage: deepeval [OPTIONS] COMMAND [ARGS]...                                   
                                                                               
┌─ Options ───────────────────────────────────────────────────────────────────┐
│ --install-completion          Install completion for the current shell.     │
│ --show-completion             Show completion for the current shell, to     │
│                               copy it or customize the installation.        │
│ --help                        Show this message and exit.                   │
└─────────────────────────────────────────────────────────────────────────────┘
┌─ Commands ──────────────────────────────────────────────────────────────────┐
│ set-confident-region           Set the Confident AI data region.            │
│ login                                                                       │
│ logout                         Log out

### Writing simple DeepEval Test

### Answer Relevancy Metrics - Standalone

In [46]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric

answer_relevancy_metric = AnswerRelevancyMetric()
test_case = LLMTestCase(
  input="Who is the current president of the United States of America?",
  actual_output="Joe Biden",
  retrieval_context=["Joe Biden serves as the current president of America."]
)

answer_relevancy_metric.measure(test_case)
print(answer_relevancy_metric.score)

1.0


[Confident AI Metric Data Log] Successfully posted metric data (0 metrics 
remaining in queue, 1 in flight) 
To disable dev logging, set CONFIDENT_METRIC_LOGGING_VERBOSE=0 as an 
environment variable.


### Test using Contextual Precision Metrics - Standalone

In [9]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualPrecisionMetric

contextual_precision_metrics = ContextualPrecisionMetric()

test_case = LLMTestCase(
    input="Who is the current president of USA in 2025",
    # Should come from an LLM or from an Agent or RAG
    actual_output="Donald Trump",
    # RAG - Vector DB, AI Agent - Agent Tools, LLM - LLM invoke response
    retrieval_context=["Donald Trump serves as the current president of America."],
    expected_output="Donald Trump is the current president of America."
)

contextual_precision_metrics.measure(test_case=test_case)
print(contextual_precision_metrics.score)
print(contextual_precision_metrics.success)
print(contextual_precision_metrics.score_breakdown)



Output()

[Confident AI Metric Data Log] Error posting metric data (0 metrics remaining in queue, 1 in flight): Invalid API 
key 
To disable dev logging, set CONFIDENT_METRIC_LOGGING_VERBOSE=0 as an environment variable.

1.0
True
None


### Answer Relevancy Metrics & Contextual Precision Metric - Combined

In [10]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric, ContextualPrecisionMetric

test_case = LLMTestCase(
    input="Can I return shoes that don’t fit?",
    actual_output="Yes, returns are possible within 30 days.",
    retrieval_context=[
        "Returns allowed for up to 30 days.",
        "Free shipping available.",
        "Shoes exchange possible within 14 days."

    ],
    expected_output="Yes, returns are possible within 30 days."
)

# Evaluate if answer matches question
answer_metric = AnswerRelevancyMetric(threshold=0.7)
print("AnswerRelevancyMetric", answer_metric.measure(test_case))

# Evaluate if relevant context is prioritized
context_metric = ContextualPrecisionMetric()
print("ContextualPrecisionMetric", context_metric.measure(test_case))


Output()

Output()

AnswerRelevancyMetric 1.0


[Confident AI Metric Data Log] Error posting metric data (0 metrics remaining in queue, 1 in flight): Invalid API 
key 
To disable dev logging, set CONFIDENT_METRIC_LOGGING_VERBOSE=0 as an environment variable.

ContextualPrecisionMetric 1.0


### Evaluate our Tests without Standalone using - Evaluate

In [20]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate

answer_relevancy_metric = AnswerRelevancyMetric()
test_case = LLMTestCase(
  input="Who is the current president of the United States of America?",
  actual_output="Joe Biden",
  retrieval_context=["Joe Biden serves as the current president of America."]
)

evaluate(test_cases=[test_case], metrics=[answer_relevancy_metric])

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1, strict=False, async_mode=True)...

Output()

INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases




Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1, reason: The score is 1.00 because the answer was fully relevant and directly addressed the question without any irrelevant information. Great job staying focused and concise!, error: None)

For test case:

  - input: Who is the current president of the United States of America?
  - actual output: Joe Biden
  - expected output: None
  - context: None
  - retrieval context: ['Joe Biden serves as the current president of America.']


Overall Metric Pass Rates

Answer Relevancy: 100.00% pass rate




⚠ WARNING: No hyperparameters logged.
» ]8;id=935268;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=967420;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhef0oob0i60la0g3fy8mdso/test-cases\https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhef0oob0i60la0g3fy8mdso/test-cases]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the answer was fully relevant and directly addressed the question without any irrelevant information. Great job staying focused and concise!', strict_mode=False, evaluation_model='gpt-4.1', error=None, evaluation_cost=0.00277, verbose_logs='Statements:\n[\n    "Joe Biden"\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]')], conversational=False, multimodal=False, input='Who is the current president of the United States of America?', actual_output='Joe Biden', expected_output=None, context=None, retrieval_context=['Joe Biden serves as the current president of America.'], turns=None, additional_metadata=None)], confident_link='https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhef0oob0i60la0g3fy8mdso/test-cases', test_run_id='cm

In [21]:
### This command tries to open the latest test run in your browser.
### If the test run is not cached locally or has errors, you may need to rerun the test after logging in again.

!deepeval view

🔗 View test run: 
https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhef0o
ob0i60la0g3fy8mdso/test-cases


### Evaluating Multiple Test Cases for Answer Relevance Metrics

In [47]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate

answer_relevancy_metric = AnswerRelevancyMetric()

test_case1 = LLMTestCase(
  input="Who is the current president of the United States of America?",
  actual_output="Joe Biden",
  retrieval_context=["Joe Biden serves as the current president of America."]
)

test_case2 = LLMTestCase(
  input="Who built the Claude Models?",
  actual_output="OpenAI",
  expected_output= "Claude Anthrophic",
  retrieval_context=["Claude Anthrophic built the Claude models."]
)

evaluate(test_cases=[test_case1, test_case2], metrics=[answer_relevancy_metric])

✨ You're running DeepEval's latest Answer Relevancy Metric! (using openai/gpt-oss-20b (Local Model), strict=False,
async_mode=True)...



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: openai/gpt-oss-20b (Local Model), reason: The score is 1.00 because the answer correctly identifies the current president of the United States of America and contains no irrelevant statements., error: None)

For test case:

  - input: Who is the current president of the United States of America?
  - actual output: Joe Biden
  - expected output: None
  - context: None
  - retrieval context: ['Joe Biden serves as the current president of America.']


Metrics Summary

  - ❌ Answer Relevancy (score: 0.0, threshold: 0.5, strict: False, evaluation model: openai/gpt-oss-20b (Local Model), reason: The score is 0.00 because the assistant failed to answer the question 'Who built the Claude Models?' and instead made unrelated statements about OpenAI and JSON, which do not provide any information about the actual builder of the Claude Models., error: None)

For test case:

  - input: Who built t

⚠ WARNING: No hyperparameters logged.
» ]8;id=702424;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=251220;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhfebb1p0dloo80g40eder7x/regression-testing\https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhfebb1p0dloo80g40eder7x/regression-testi]8;;\
]8;id=251220;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhfebb1p0dloo80g40eder7x/regression-testing\ng]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the answer correctly identifies the current president of the United States of America and contains no irrelevant statements.', strict_mode=False, evaluation_model='openai/gpt-oss-20b (Local Model)', error=None, evaluation_cost=0.0, verbose_logs='Statements:\n[\n    "Joe Biden"\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]')], conversational=False, multimodal=False, input='Who is the current president of the United States of America?', actual_output='Joe Biden', expected_output=None, context=None, retrieval_context=['Joe Biden serves as the current president of America.'], turns=None, additional_metadata=None), TestResult(name='test_case_1', success=False, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=False, score=0.0, re

### Evaluate With Golden DataSet and EvaluationDataSet

In [10]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate
from deepeval.dataset import EvaluationDataset, Golden

# Create Golden instead of Test cases
golden = Golden(
    input="Who is the current president of the United States of America?",
    expected_output="Joe Biden",
    context=["Joe Biden serves as the current president of America."]
)

dataset = EvaluationDataset()
dataset.add_golden(golden)



In [11]:
dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='Who is the current president of the United States of America?', actual_output=None, expected_output='Joe Biden', context=['Joe Biden serves as the current president of America.'], retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None)], _alias=None, _id=None, _multi_turn=False)

#### Creating Test Case from Golden

In [12]:
for golden in dataset.goldens:
    test_case = LLMTestCase(
        input=golden.input,                     ### User Input
        expected_output=golden.expected_output,  ### Ground Truth
        actual_output="Joe Biden",   ### LLM Output
        retrieval_context=golden.context  ### Retriever Output
    )

    dataset.add_test_case(test_case)



In [13]:
dataset

EvaluationDataset(test_cases=[LLMTestCase(input='Who is the current president of the United States of America?', actual_output='Joe Biden', expected_output='Joe Biden', context=None, retrieval_context=['Joe Biden serves as the current president of America.'], additional_metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None)], goldens=[Golden(input='Who is the current president of the United States of America?', actual_output=None, expected_output='Joe Biden', context=['Joe Biden serves as the current president of America.'], retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None)], _alias=None, _id=None, _multi_turn=False)

In [14]:
evaluate(test_cases=dataset.test_cases, metrics=[AnswerRelevancyMetric()])

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1, strict=False, async_mode=True)...

Output()

INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases




Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1, reason: The score is 1.00 because the answer was fully relevant and directly addressed the question without any irrelevant information. Great job staying focused and concise!, error: None)

For test case:

  - input: Who is the current president of the United States of America?
  - actual output: Joe Biden
  - expected output: Joe Biden
  - context: None
  - retrieval context: ['Joe Biden serves as the current president of America.']


Overall Metric Pass Rates

Answer Relevancy: 100.00% pass rate




⚠ WARNING: No hyperparameters logged.
» ]8;id=84711;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=560382;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmherf1nn01l7o80g8oiuifhi/regression-testing\https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmherf1nn01l7o80g8oiuifhi/regression-testi]8;;\
]8;id=560382;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmherf1nn01l7o80g8oiuifhi/regression-testing\ng]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the answer was fully relevant and directly addressed the question without any irrelevant information. Great job staying focused and concise!', strict_mode=False, evaluation_model='gpt-4.1', error=None, evaluation_cost=0.0027800000000000004, verbose_logs='Statements:\n[\n    "Joe Biden."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]')], conversational=False, multimodal=False, input='Who is the current president of the United States of America?', actual_output='Joe Biden', expected_output='Joe Biden', context=None, retrieval_context=['Joe Biden serves as the current president of America.'], turns=None, additional_metadata=None)], confident_link='https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmherf1nn01l7o80g8oiuifhi/regres

#### Creating Evaluation Dataset as Goldens in Confident AI

###### Creating Data

In [15]:
test_data = [
    {
        "input": "Who is the current president of the United States of America?",
        "expected_output": "Joe Biden",
    },
    {
        "input": "Who introducted the GPT Model?",
        "expected_output": "Open AI"
    }
]

In [16]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate
from deepeval.dataset import EvaluationDataset, Golden

goldens = []

for data in test_data:
    golden = Golden(
        input= data['input'],
        expected_output=data['expected_output'],
    )
    goldens.append(golden)

new_dataset = EvaluationDataset(goldens=goldens)
new_dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='Who is the current president of the United States of America?', actual_output=None, expected_output='Joe Biden', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None), Golden(input='Who introducted the GPT Model?', actual_output=None, expected_output='Open AI', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None)], _alias=None, _id=None, _multi_turn=False)

#### Push the Dataset to Confident AI

In [18]:
new_dataset.push(alias="TestGoldenDataSet")

✅ Dataset successfully pushed to Confident AI! View at 
]8;id=689841;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/datasets/cmhet4lgn038ko80gxavtp179\https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/datasets/cmhet4lgn038ko80gxavtp179]8;;\

In [19]:
new_dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='Who is the current president of the United States of America?', actual_output=None, expected_output='Joe Biden', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None), Golden(input='Who introducted the GPT Model?', actual_output=None, expected_output='Open AI', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None)], _alias=None, _id=None, _multi_turn=False)

#### Pull the Dataset from Confident AI


In [26]:
cloudDataSet = EvaluationDataset()
cloudDataSet.pull(alias="TestGoldenDataSet")
cloudDataSet

Output()

EvaluationDataset(test_cases=[], goldens=[Golden(input='Who is the current president of the United States of America?', actual_output=None, expected_output='Modi', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None), Golden(input='Who introducted the GPT Model?', actual_output=None, expected_output='Open AI', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None)], _alias=TestGoldenDataSet, _id=cmhet4lgn038ko80gxavtp179, _multi_turn=False)

#### Prepare our Testcase to evaluate our records

In [27]:
def mock_llms_app(input):
    if input == 1:
        return "Joe Biden"
    elif input == 2:
        return "Open AI"

In [28]:
from deepeval.test_case import LLMTestCase

counter = 1
for golden in cloudDataSet.goldens:
    test_case = LLMTestCase(
        input= golden.input,                     ### User Input
        expected_output=golden.expected_output,  ### Ground Truth
        actual_output=mock_llms_app(counter),    ### Output from LLM
    )
    counter += 1
    cloudDataSet.add_test_case(test_case)

In [29]:
print(cloudDataSet.test_cases)

[LLMTestCase(input='Who is the current president of the United States of America?', actual_output='Joe Biden', expected_output='Modi', context=None, retrieval_context=None, additional_metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None), LLMTestCase(input='Who introducted the GPT Model?', actual_output='Open AI', expected_output='Open AI', context=None, retrieval_context=None, additional_metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None)]


In [30]:
print(cloudDataSet.goldens)

[Golden(input='Who is the current president of the United States of America?', actual_output=None, expected_output='Modi', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None), Golden(input='Who introducted the GPT Model?', actual_output=None, expected_output='Open AI', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None)]


In [31]:
evaluate(test_cases=cloudDataSet.test_cases, metrics=[AnswerRelevancyMetric()])

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1, strict=False, async_mode=True)...

Output()

INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases




Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1, reason: The score is 1.00 because the answer was fully relevant and directly addressed the question without any irrelevant information. Great job staying focused and concise!, error: None)

For test case:

  - input: Who is the current president of the United States of America?
  - actual output: Joe Biden
  - expected output: Modi
  - context: None
  - retrieval context: None


Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1, reason: The score is 1.00 because the answer was fully relevant and directly addressed the question without any irrelevant information. Great job staying focused and concise!, error: None)

For test case:

  - input: Who introducted the GPT Model?
  - actual output: Open AI
  - expected output: Open AI
  - context: None
  - retrieval context: None


Overall Metric Pass Rates

Answer Relevanc

⚠ WARNING: No hyperparameters logged.
» ]8;id=535380;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=524871;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmheucbr60589o80grxx5u1zg/regression-testing\https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmheucbr60589o80grxx5u1zg/regression-testi]8;;\
]8;id=524871;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmheucbr60589o80grxx5u1zg/regression-testing\ng]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the answer was fully relevant and directly addressed the question without any irrelevant information. Great job staying focused and concise!', strict_mode=False, evaluation_model='gpt-4.1', error=None, evaluation_cost=0.002746, verbose_logs='Statements:\n[\n    "Joe Biden"\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]')], conversational=False, multimodal=False, input='Who is the current president of the United States of America?', actual_output='Joe Biden', expected_output='Modi', context=None, retrieval_context=None, turns=None, additional_metadata=None), TestResult(name='test_case_1', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the answer was fully rele

### Using Local LLM for Evaluation

In [ ]:
### In case if you want to reset the mapping between Deepeval and Ollama models

# !deepeval unset-ollama

🙌 OpenAI will still be used by default because OPENAI_API_KEY is set.


In [ ]:
### Use the below command to set the Ollama model with Deepeval. Also ensure that .deepeval file should contain "LOCAL_MODEL_API_KEY": "ollama"

# !deepeval set-ollama llama3.2:latest
# !deepeval set-ollama llama3.2:latest --base-url "http://localhost:11434"

Settings updated for this session. To persist, use --save=dotenv[:path] 
(default .env.local) or set DEEPEVAL_DEFAULT_SAVE=dotenv:.env.local
🙌 Congratulations! You're now using a local Ollama model `llama3.2:latest` for
all evals that require an LLM.


### Instead of Ollama if you want to use Groq API as you LLM provider for Deepeval then please perform the below configuation

In [ ]:
import os
os.environ["OPENAI_BASE_URL"] = ""  # Groq’s OpenAI-compatible API
os.environ["OPENAI_API_KEY"] = ""               # use your Groq key here


In [ ]:
!deepeval set-local-model --model-name="openai/gpt-oss-20b" --base-url="" --api-key=""


Settings updated for this session. To persist, use --save=dotenv[:path] 
(default .env.local) or set DEEPEVAL_DEFAULT_SAVE=dotenv:.env.local
🙌 Congratulations! You're now using a local model `openai/gpt-oss-20b` for all
evals that require an LLM.


In [ ]:
os.environ["OPENAI_API_KEY"]

In [45]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate

answer_relevancy_metric = AnswerRelevancyMetric(model="openai/gpt-oss-20b")

test_case1 = LLMTestCase(
  input="Who is the current president of the United States of America?",
  actual_output="Joe Biden",
  retrieval_context=["Joe Biden serves as the current president of America."]
)

test_case2 = LLMTestCase(
  input="Who built the Claude Models?",
  actual_output="OpenAI",
  expected_output= "Claude Anthrophic",
  retrieval_context=["Claude Anthrophic built the GPT models."]
)

evaluate(test_cases=[test_case1, test_case2], metrics=[answer_relevancy_metric])

✨ You're running DeepEval's latest Answer Relevancy Metric! (using openai/gpt-oss-20b (Local Model), strict=False,
async_mode=True)...



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: openai/gpt-oss-20b (Local Model), reason: The score is 1.00 because the response directly addressed the question and contained no irrelevant statements; it is already at the maximum possible score., error: None)

For test case:

  - input: Who is the current president of the United States of America?
  - actual output: Joe Biden
  - expected output: None
  - context: None
  - retrieval context: ['Joe Biden serves as the current president of America.']


Metrics Summary

  - ❌ Answer Relevancy (score: 0.0, threshold: 0.5, strict: False, evaluation model: openai/gpt-oss-20b (Local Model), reason: The score is 0.00 because the answer includes an irrelevant statement about OpenAI not building the Claude Models, which does not directly address the question of who built the Claude Models., error: None)

For test case:

  - input: Who built the Claude Models?
  - actual output: OpenAI
  - e

⚠ WARNING: No hyperparameters logged.
» ]8;id=225651;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=321983;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhfdzj8b0d8go80gwhekyfif/regression-testing\https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhfdzj8b0d8go80gwhekyfif/regression-testi]8;;\
]8;id=321983;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhfdzj8b0d8go80gwhekyfif/regression-testing\ng]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the response directly addressed the question and contained no irrelevant statements; it is already at the maximum possible score.', strict_mode=False, evaluation_model='openai/gpt-oss-20b (Local Model)', error=None, evaluation_cost=0.0, verbose_logs='Statements:\n[\n    "Joe Biden"\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]')], conversational=False, multimodal=False, input='Who is the current president of the United States of America?', actual_output='Joe Biden', expected_output=None, context=None, retrieval_context=['Joe Biden serves as the current president of America.'], turns=None, additional_metadata=None), TestResult(name='test_case_1', success=False, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=False, score=0.